# stage 1: Classical Selection

**Objective:** build a reference answer for selection that i can check independently, test the three-comparator median network exhaustively, and reproduce the 3 by 3 by 3 counterexample from the cuboid manuscript.

## 1. Setup

record of environment so results can be reproduced

In [ ]:
import sys, platform
sys.path.append("../src")

import numpy, matplotlib, pennylane
print("Python    ", platform.python_version())
print("numpy     ", numpy.__version__)
print("matplotlib", matplotlib.__version__)
print("pennylane ", pennylane.__version__)

## 2. Conventions

- Ranks are one-based. Rank 1 is the smallest element, so rank k is `sorted(x)[k-1]` in python.
- Integers are unsigned, with bits written most significant first
- even N: the target is the *lower median*, k = ceil(N/2)
- grid coordinates in the manuscript are one-based and numpy is zero-based: manuscript's position (1,3,3) is `A[0][2][2].

In [ ]:
from selection import lower_median_rank

for N in [1, 2, 3, 4, 5, 6, 27]:
    print(f"N = {N:2d}  ->  k = {lower_median_rank(N)}")

## 3. The three-comparator median network

Compare-exchange on wires (0,1), then (1,2), then (0,1).

Why it works:
After the first two comparisons, the largest value has moved to
wire 2. The last comparison orders the remaining two values on wires 0 and 1.
The list is then fully sorted, so wire 1 holds the median. It always uses
exactly three comparisons, regardless of input.

In [ ]:
from draw_network import draw
from IPython.display import Image

draw([(0, 1), (1, 2), (0, 1)], 3, "../results/median3_network.png")
Image("../results/median3_network.png")

## 4. Exhaustive test

I test every ordered triple of values in {0,1,2,3}, which is 4^3 = 64 cases and
includes repeated values (Three *distinct* 2-bit values would give only 24
cases and would miss ties.) The network's output is compared with the
independent reference `sorted(x)[1]`. I also check that exactly three
comparisons are used, counted separately from run time.

In [ ]:
from itertools import product
from selection import median3, reference_select

failures = [x for x in product(range(4), repeat=3)
            if median3(x) != reference_select(x, 2)]
print(f"{64 - len(failures)}/64 passed. Failing inputs: {failures}")

counts = set()
for x in product(range(4), repeat=3):
    c = []
    median3(x, counter=c)
    counts.add(len(c))
print("Comparisons used across all inputs:", counts)

## 5. The 3x3x3 counterexample

From the manuscript (Section VI-C, "An explicit fixed-point counterexample").
The array uses the values 1 to 27. Rows are indexed by j, columns by k, and
each slice is one value of i.

In [ ]:
from counterexample import A, is_coordinatewise_increasing, median_position

for i in range(3):
    print(f"i = {i+1}")
    print(A[i], "\n")

print("Fixed point (no inverted neighbour along any axis):",
      is_coordinatewise_increasing(A))

m, pos = median_position(A)
print("Median:", m, "at (i,j,k) =", pos)
print("Central plane k=2 values:", sorted(A[:, :, 1].flatten()))
print("Median inside central plane:", m in A[:, :, 1])

**What this shows.**
With exact comparisons, a compare-exchange only swaps a
pair that is out of order. No neighbouring pair here is out of order, so the
noiseless schedule never changes this array, for any number of iterations or
any random sampling of edges. Yet the median (14) is not in the central plane
k = 2. This shows that a central plane can miss the median, and local ordering is not
selection.

**Caveat (from the manuscript).** 
This is for exact comparisons only. It does
not claim the array stays fixed when comparisons are noisy.

## 6. Candidate sets and rank

Knowing the target is *in* a candidate set does not tell me its rank *within*
that set. Example: X = {1,...,9} has median 5. The candidate set
L = {1, 2, 5} contains 5, but the median of L is 2.

In [ ]:
X = list(range(1, 10))
L_set = [1, 2, 5]
print("Median of X:           ", reference_select(X, 5))
print("5 is in the candidates:", 5 in L_set)
print("Median of candidate set:", reference_select(L_set, 2))

To use a candidate set I need a certificate. Split X into B (below), L
(candidates) and U (above) so that every element of B is smaller than every
element of L, which is smaller than every element of U. If |B| < k <= |B|+|L|,
then the k-th element of X is the (k - |B|)-th element of L. The adjusted
rank k' = k - |B| is what must be passed on, not the middle of L.
(Manuscript, Proposition 7.)

## 9. Summary

- The three-comparator network gave the correct median on all [64/64] ordered
  triples, using exactly three comparisons each.
- I reproduced the 3x3x3 counterexample: it is a fixed point of the noiseless
  schedule and its median (14, at (1,3,3)) lies outside the central plane.
- Containing the target in a candidate set does not determine its rank in
  that set: a certificate and an adjusted rank are needed.
- The quantum work starts from the correct network, with the cuboid kept
  as motivation.

